In [1]:
import pandas as pd
import os

base_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset"

print(os.listdir(base_path))

['Faisalabad', 'Islamabad', 'Lahore', 'master_dataset.csv', 'Multan']


In [2]:
all_dfs = []

cities = ['Lahore','Islamabad','Faisalabad','Multan']

for city in cities:
    city_path = os.path.join(base_path, city)
    for file in os.listdir(city_path):
        if file.endswith('.csv'):
            filepath = os.path.join(city_path, file)
            df = pd.read_csv(filepath)
            df['city'] = city
            all_dfs.append(df)
            print(f"Loaded: {file}-{len(df)} rows")

master_df = pd.concat(all_dfs, ignore_index = True)
print(f"\nTotal rows in master dataset: {len(master_df)}")
print(f"Total columns: {len(master_df.columns)}")

Loaded: Lahore_2021_2022.csv-2880 rows
Loaded: Lahore_2022_2023.csv-2880 rows
Loaded: Lahore_2023_2024.csv-2904 rows
Loaded: Lahore_2024_2025.csv-2880 rows
Loaded: Islamabad_2021_2022.csv-2880 rows
Loaded: Islamabad_2022_2023.csv-2880 rows
Loaded: Islamabad_2023_2024.csv-2904 rows
Loaded: Islamabad_2024_2025.csv-2880 rows
Loaded: Faisalabad_2021_2022.csv-2880 rows
Loaded: Faisalabad_2022_2023.csv-2880 rows
Loaded: Faisalabad_2023_2024.csv-2904 rows
Loaded: Faisalabad_2024_2025.csv-2880 rows
Loaded: Multan_2021_2022.csv-2880 rows
Loaded: Multan_2022_2023.csv-2880 rows
Loaded: Multan_2023_2024.csv-2904 rows
Loaded: Multan_2024_2025.csv-2880 rows

Total rows in master dataset: 46176
Total columns: 25


In [3]:
print(master_df.columns.tolist())

['name', 'datetime', 'temp', 'feelslike', 'dew', 'humidity', 'precip', 'precipprob', 'preciptype', 'snow', 'snowdepth', 'windgust', 'windspeed', 'winddir', 'sealevelpressure', 'cloudcover', 'visibility', 'solarradiation', 'solarenergy', 'uvindex', 'severerisk', 'conditions', 'icon', 'stations', 'city']


In [4]:
columns_to_keep = [
    'datetime', 'city', 'temp', 'humidity', 'dew',
    'windspeed', 'windgust', 'winddir', 'sealevelpressure',
    'cloudcover', 'visibility', 'precip', 'snow', 'snowdepth'
]

master_df = master_df[columns_to_keep]

print(f"Columns kept: {master_df.columns.tolist()}")
print(f"Shape: {master_df.shape}")

Columns kept: ['datetime', 'city', 'temp', 'humidity', 'dew', 'windspeed', 'windgust', 'winddir', 'sealevelpressure', 'cloudcover', 'visibility', 'precip', 'snow', 'snowdepth']
Shape: (46176, 14)


In [5]:
print("Missing values in each column:")
print(master_df.isnull().sum())

Missing values in each column:
datetime               0
city                   0
temp                   0
humidity               0
dew                    0
windspeed              0
windgust               4
winddir                0
sealevelpressure       0
cloudcover             0
visibility          5192
precip                 1
snow                   0
snowdepth              0
dtype: int64


In [6]:
master_df['windgust'] = master_df['windgust'].fillna(master_df['windgust'].median())

master_df['precip'] = master_df['precip'].fillna(0)

master_df['visibility'] = master_df.groupby('city')['visibility'].transform(
lambda x: x.ffill().bfill())

print("Missing values after filling:")
print(master_df.isnull().sum())

Missing values after filling:
datetime            0
city                0
temp                0
humidity            0
dew                 0
windspeed           0
windgust            0
winddir             0
sealevelpressure    0
cloudcover          0
visibility          0
precip              0
snow                0
snowdepth           0
dtype: int64


In [7]:

master_df['datetime'] = pd.to_datetime(master_df['datetime'])
master_df = master_df.sort_values(
    ['city', 'datetime']).reset_index(drop=True)

output_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\master_dataset.csv"
master_df.to_csv(output_path, index=False)

print("Master dataset saved successfully!")
print(f"Final shape: {master_df.shape}")
print(master_df.head(3))

Master dataset saved successfully!
Final shape: (46176, 14)
             datetime        city  temp  humidity   dew  windspeed  windgust  \
0 2021-11-01 00:00:00  Faisalabad  20.6     64.19  13.6        7.9      13.0   
1 2021-11-01 01:00:00  Faisalabad  19.5     66.94  13.2        8.3      13.3   
2 2021-11-01 02:00:00  Faisalabad  17.6     70.62  12.2        0.0      14.0   

   winddir  sealevelpressure  cloudcover  visibility  precip  snow  snowdepth  
0    113.8            1012.0         0.0         4.0     0.0     0        0.0  
1    107.5            1012.0         0.0         4.0     0.0     0        0.0  
2    360.0            1011.9         0.0         4.0     0.0     0        0.0  


In [16]:
missing_vis = master_df[master_df['visibility'].isnull()]

print("Total missing visibility:", len(missing_vis))
print("\nMissing by City:")
print(missing_vis['city'].value_counts())
print("\nMissing by Month:")
print(missing_vis['datetime'].dt.month.value_counts())

Total missing visibility: 0

Missing by City:
Series([], Name: count, dtype: int64)

Missing by Month:
Series([], Name: count, dtype: int64)


In [18]:
test_file = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\Lahore\Lahore_2021_2022.csv"
test_df = pd.read_csv(test_file)
print("Missing visibility in Lahore 2021-2022:", test_df['visibility'].isnull().sum())
print("Total rows:" , len(test_df))

Missing visibility in Lahore 2021-2022: 2
Total rows: 2880


In [20]:
all_dfs = []
cities = ['Lahore', 'Islamabad', 'Faisalabad', 'Multan']

for city in cities:
    city_path = os.path.join(base_path, city)
    for file in os.listdir(city_path):
        if file.endswith('.csv'):
            filepath = os.path.join(city_path, file)
            df = pd.read_csv(filepath)
            df['city'] = city
            all_dfs.append(df)

master_df = pd.concat(all_dfs, ignore_index=True)
master_df = master_df[columns_to_keep]
master_df['datetime'] = pd.to_datetime(master_df['datetime'])

# NOW drop missing visibility rows
before = len(master_df)
master_df = master_df.dropna(subset=['visibility'])
after = len(master_df)

print(f"Rows before: {before}")
print(f"Rows after dropping: {after}")
print(f"Rows dropped: {before - after}")

Rows before: 46176
Rows after dropping: 40984
Rows dropped: 5192


In [22]:
master_df['windgust'] = master_df['windgust'].fillna(
    master_df['windgust'].median()
)
master_df['precip'] = master_df['precip'].fillna(0)
print("Missing values after cleaning:")
print(master_df.isnull().sum())
output_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\master_dataset.csv"
master_df.to_csv(output_path, index=False)
print("\nFinal dataset saved!")
print(f"Final shape: {master_df.shape}")

Missing values after cleaning:
datetime            0
city                0
temp                0
humidity            0
dew                 0
windspeed           0
windgust            0
winddir             0
sealevelpressure    0
cloudcover          0
visibility          0
precip              0
snow                0
snowdepth           0
dtype: int64

Final dataset saved!
Final shape: (40984, 14)
